# Project 6: Enterprise Retail Data Warehouse - Snowflake Schema

In [ ]:
%%sql -r setup_db
CREATE DATABASE IF NOT EXISTS RETAIL_SNOWFLAKE_DW;
CREATE SCHEMA IF NOT EXISTS RETAIL_SNOWFLAKE_DW.SNOWFLAKE_SCHEMA;



In [ ]:
%%sql -r dataframe_1
USE DATABASE RETAIL_SNOWFLAKE_DW;
USE SCHEMA SNOWFLAKE_SCHEMA;

## Lookup Tables (Hierarchical - Top Level First)
### Date Hierarchy: DIM_YEAR → DIM_QUARTER → DIM_MONTH

In [ ]:
%%sql -r create_dim_year
CREATE OR REPLACE TABLE DIM_YEAR (
    year_id INT PRIMARY KEY,
    year_value INT NOT NULL
);

In [ ]:
%%sql -r create_dim_quarter
CREATE OR REPLACE TABLE DIM_QUARTER (
    quarter_id INT PRIMARY KEY,
    quarter_name VARCHAR(10) NOT NULL,
    year_id INT NOT NULL,
    FOREIGN KEY (year_id) REFERENCES DIM_YEAR(year_id)
);

In [ ]:
%%sql -r create_dim_month
CREATE OR REPLACE TABLE DIM_MONTH (
    month_id INT PRIMARY KEY,
    month_name VARCHAR(20) NOT NULL,
    quarter_id INT NOT NULL,
    FOREIGN KEY (quarter_id) REFERENCES DIM_QUARTER(quarter_id)
);

### Product Hierarchy: DIM_CATEGORY → DIM_BRAND

In [ ]:
%%sql -r create_dim_category
CREATE OR REPLACE TABLE DIM_CATEGORY (
    category_id INT PRIMARY KEY,
    category_name VARCHAR(100) NOT NULL
);

In [ ]:
%%sql -r create_dim_brand
CREATE OR REPLACE TABLE DIM_BRAND (
    brand_id INT PRIMARY KEY,
    brand_name VARCHAR(100) NOT NULL,
    category_id INT NOT NULL,
    FOREIGN KEY (category_id) REFERENCES DIM_CATEGORY(category_id)
);

### Location Hierarchy: DIM_REGION → DIM_STATE → DIM_CITY

In [ ]:
%%sql -r create_dim_region
CREATE OR REPLACE TABLE DIM_REGION (
    region_id INT PRIMARY KEY,
    region_name VARCHAR(50) NOT NULL
);

In [ ]:
%%sql -r create_dim_state
CREATE OR REPLACE TABLE DIM_STATE (
    state_id INT PRIMARY KEY,
    state_name VARCHAR(100) NOT NULL,
    region_id INT NOT NULL,
    FOREIGN KEY (region_id) REFERENCES DIM_REGION(region_id)
);

In [ ]:
%%sql -r create_dim_city
CREATE OR REPLACE TABLE DIM_CITY (
    city_id INT PRIMARY KEY,
    city_name VARCHAR(100) NOT NULL,
    state_id INT NOT NULL,
    FOREIGN KEY (state_id) REFERENCES DIM_STATE(state_id)
);

## Normalized Dimension Tables

In [ ]:
%%sql -r create_dim_customer
CREATE OR REPLACE TABLE DIM_CUSTOMER (
    customer_id INT PRIMARY KEY,
    customer_name VARCHAR(100) NOT NULL,
    city_id INT NOT NULL,
    membership VARCHAR(20) NOT NULL,
    FOREIGN KEY (city_id) REFERENCES DIM_CITY(city_id)
);

In [ ]:
%%sql -r create_dim_product
CREATE OR REPLACE TABLE DIM_PRODUCT (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    brand_id INT NOT NULL,
    price DECIMAL(10,2) NOT NULL,
    FOREIGN KEY (brand_id) REFERENCES DIM_BRAND(brand_id)
);

In [ ]:
%%sql -r create_dim_branch
CREATE OR REPLACE TABLE DIM_BRANCH (
    branch_id INT PRIMARY KEY,
    branch_name VARCHAR(100) NOT NULL,
    city_id INT NOT NULL,
    manager_name VARCHAR(100) NOT NULL,
    FOREIGN KEY (city_id) REFERENCES DIM_CITY(city_id)
);

In [ ]:
%%sql -r create_dim_date
CREATE OR REPLACE TABLE DIM_DATE (
    date_id INT PRIMARY KEY,
    date DATE NOT NULL,
    day INT NOT NULL,
    day_name VARCHAR(20) NOT NULL,
    week_no INT NOT NULL,
    month_id INT NOT NULL,
    is_weekend VARCHAR(3) NOT NULL,
    FOREIGN KEY (month_id) REFERENCES DIM_MONTH(month_id)
);

## Fact Table

In [ ]:
%%sql -r create_fact_sales
CREATE OR REPLACE TABLE FACT_SALES (
    sale_id INT PRIMARY KEY,
    customer_id INT NOT NULL,
    product_id INT NOT NULL,
    branch_id INT NOT NULL,
    date_id INT NOT NULL,
    quantity INT NOT NULL,
    total_amount DECIMAL(12,2) NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES DIM_CUSTOMER(customer_id),
    FOREIGN KEY (product_id) REFERENCES DIM_PRODUCT(product_id),
    FOREIGN KEY (branch_id) REFERENCES DIM_BRANCH(branch_id),
    FOREIGN KEY (date_id) REFERENCES DIM_DATE(date_id)
);

## Phase 5: Data Loading
*(Upload CSV files to stage and load data here)*

## Phase 6: Validation Queries

In [ ]:
%%sql -r query1_customer_sales
-- Query 1: Customer-wise Sales Report
SELECT c.customer_name, ci.city_name, s.state_name, c.membership,
    COUNT(f.sale_id) AS total_transactions, SUM(f.total_amount) AS total_revenue
FROM FACT_SALES f
JOIN DIM_CUSTOMER c ON f.customer_id = c.customer_id
JOIN DIM_CITY ci ON c.city_id = ci.city_id
JOIN DIM_STATE s ON ci.state_id = s.state_id
GROUP BY c.customer_name, ci.city_name, s.state_name, c.membership
ORDER BY total_revenue DESC;

In [ ]:
%%sql -r query2_product_revenue
-- Query 2: Product-wise Revenue Report
SELECT p.product_name, br.brand_name, cat.category_name, p.price,
    SUM(f.quantity) AS total_units, SUM(f.total_amount) AS total_revenue
FROM FACT_SALES f
JOIN DIM_PRODUCT p ON f.product_id = p.product_id
JOIN DIM_BRAND br ON p.brand_id = br.brand_id
JOIN DIM_CATEGORY cat ON br.category_id = cat.category_id
GROUP BY p.product_name, br.brand_name, cat.category_name, p.price
ORDER BY total_revenue DESC;

In [ ]:
%%sql -r query3_brand_revenue
-- Query 3: Brand-wise Revenue Report
SELECT br.brand_name, cat.category_name,
    SUM(f.total_amount) AS total_revenue, SUM(f.quantity) AS total_units
FROM FACT_SALES f
JOIN DIM_PRODUCT p ON f.product_id = p.product_id
JOIN DIM_BRAND br ON p.brand_id = br.brand_id
JOIN DIM_CATEGORY cat ON br.category_id = cat.category_id
GROUP BY br.brand_name, cat.category_name
ORDER BY total_revenue DESC;

In [ ]:
%%sql -r query4_category_revenue
-- Query 4: Category-wise Revenue Report
SELECT cat.category_name,
    COUNT(f.sale_id) AS total_transactions,
    SUM(f.quantity) AS total_units,
    SUM(f.total_amount) AS total_revenue
FROM FACT_SALES f
JOIN DIM_PRODUCT p ON f.product_id = p.product_id
JOIN DIM_BRAND br ON p.brand_id = br.brand_id
JOIN DIM_CATEGORY cat ON br.category_id = cat.category_id
GROUP BY cat.category_name
ORDER BY total_revenue DESC;

In [ ]:
%%sql -r query5_region_revenue
-- Query 5: Region-wise Revenue Report
SELECT r.region_name,
    COUNT(f.sale_id) AS total_transactions,
    SUM(f.total_amount) AS total_revenue
FROM FACT_SALES f
JOIN DIM_BRANCH b ON f.branch_id = b.branch_id
JOIN DIM_CITY ci ON b.city_id = ci.city_id
JOIN DIM_STATE s ON ci.state_id = s.state_id
JOIN DIM_REGION r ON s.region_id = r.region_id
GROUP BY r.region_name
ORDER BY total_revenue DESC;

In [ ]:
%%sql -r query6_state_revenue
-- Query 6: State-wise Revenue Report
SELECT s.state_name, r.region_name,
    COUNT(f.sale_id) AS total_transactions,
    SUM(f.total_amount) AS total_revenue
FROM FACT_SALES f
JOIN DIM_BRANCH b ON f.branch_id = b.branch_id
JOIN DIM_CITY ci ON b.city_id = ci.city_id
JOIN DIM_STATE s ON ci.state_id = s.state_id
JOIN DIM_REGION r ON s.region_id = r.region_id
GROUP BY s.state_name, r.region_name
ORDER BY total_revenue DESC;

In [ ]:
%%sql -r query7_city_sales
-- Query 7: City-wise Sales Report
SELECT ci.city_name, s.state_name,
    COUNT(f.sale_id) AS total_transactions,
    SUM(f.total_amount) AS total_revenue
FROM FACT_SALES f
JOIN DIM_BRANCH b ON f.branch_id = b.branch_id
JOIN DIM_CITY ci ON b.city_id = ci.city_id
JOIN DIM_STATE s ON ci.state_id = s.state_id
GROUP BY ci.city_name, s.state_name
ORDER BY total_revenue DESC;

In [ ]:
%%sql -r query8_monthly_revenue
-- Query 8: Monthly Revenue Report (via Date Hierarchy)
SELECT m.month_name, q.quarter_name, y.year_value,
    COUNT(f.sale_id) AS total_transactions,
    SUM(f.total_amount) AS total_revenue
FROM FACT_SALES f
JOIN DIM_DATE d ON f.date_id = d.date_id
JOIN DIM_MONTH m ON d.month_id = m.month_id
JOIN DIM_QUARTER q ON m.quarter_id = q.quarter_id
JOIN DIM_YEAR y ON q.year_id = y.year_id
GROUP BY m.month_name, q.quarter_name, y.year_value;

In [ ]:
%%sql -r query9_weekly_revenue
-- Query 9: Weekly Revenue Report
SELECT d.week_no, m.month_name, q.quarter_name, y.year_value,
    SUM(f.total_amount) AS weekly_revenue
FROM FACT_SALES f
JOIN DIM_DATE d ON f.date_id = d.date_id
JOIN DIM_MONTH m ON d.month_id = m.month_id
JOIN DIM_QUARTER q ON m.quarter_id = q.quarter_id
JOIN DIM_YEAR y ON q.year_id = y.year_id
GROUP BY d.week_no, m.month_name, q.quarter_name, y.year_value
ORDER BY d.week_no;

In [ ]:
%%sql -r query10_top_customers
-- Query 10: Top 10 Customers
SELECT c.customer_name, c.membership, ci.city_name, s.state_name,
    SUM(f.total_amount) AS total_revenue
FROM FACT_SALES f
JOIN DIM_CUSTOMER c ON f.customer_id = c.customer_id
JOIN DIM_CITY ci ON c.city_id = ci.city_id
JOIN DIM_STATE s ON ci.state_id = s.state_id
GROUP BY c.customer_name, c.membership, ci.city_name, s.state_name
ORDER BY total_revenue DESC
LIMIT 10;

In [ ]:
%%sql -r query11_top_products
-- Query 11: Top 10 Products
SELECT p.product_name, br.brand_name, cat.category_name,
    SUM(f.total_amount) AS total_revenue
FROM FACT_SALES f
JOIN DIM_PRODUCT p ON f.product_id = p.product_id
JOIN DIM_BRAND br ON p.brand_id = br.brand_id
JOIN DIM_CATEGORY cat ON br.category_id = cat.category_id
GROUP BY p.product_name, br.brand_name, cat.category_name
ORDER BY total_revenue DESC
LIMIT 10;

In [ ]:
%%sql -r query12_top_branches
-- Query 12: Top 10 Branches
SELECT b.branch_name, ci.city_name, s.state_name, r.region_name,
    SUM(f.total_amount) AS total_revenue
FROM FACT_SALES f
JOIN DIM_BRANCH b ON f.branch_id = b.branch_id
JOIN DIM_CITY ci ON b.city_id = ci.city_id
JOIN DIM_STATE s ON ci.state_id = s.state_id
JOIN DIM_REGION r ON s.region_id = r.region_id
GROUP BY b.branch_name, ci.city_name, s.state_name, r.region_name
ORDER BY total_revenue DESC
LIMIT 10;

In [ ]:
%%sql -r query13_customer_trend
-- Query 13: Customer Purchase Trend (by week)
SELECT c.customer_name, d.week_no,
    COUNT(f.sale_id) AS purchases, SUM(f.total_amount) AS revenue
FROM FACT_SALES f
JOIN DIM_CUSTOMER c ON f.customer_id = c.customer_id
JOIN DIM_DATE d ON f.date_id = d.date_id
GROUP BY c.customer_name, d.week_no
ORDER BY c.customer_name, d.week_no;

In [ ]:
%%sql -r query14_weekend_analysis
-- Query 14: Weekend vs Weekday Sales Analysis
SELECT d.is_weekend,
    COUNT(f.sale_id) AS total_transactions,
    SUM(f.total_amount) AS total_revenue,
    ROUND(AVG(f.total_amount), 2) AS avg_transaction_value
FROM FACT_SALES f
JOIN DIM_DATE d ON f.date_id = d.date_id
GROUP BY d.is_weekend;

In [ ]:
%%sql -r query15_branch_performance
-- Query 15: Branch Performance with Full Location Hierarchy
SELECT b.branch_name, b.manager_name, ci.city_name, s.state_name, r.region_name,
    COUNT(f.sale_id) AS total_transactions,
    SUM(f.quantity) AS total_units,
    SUM(f.total_amount) AS total_revenue,
    ROUND(AVG(f.total_amount), 2) AS avg_sale_value
FROM FACT_SALES f
JOIN DIM_BRANCH b ON f.branch_id = b.branch_id
JOIN DIM_CITY ci ON b.city_id = ci.city_id
JOIN DIM_STATE s ON ci.state_id = s.state_id
JOIN DIM_REGION r ON s.region_id = r.region_id
GROUP BY b.branch_name, b.manager_name, ci.city_name, s.state_name, r.region_name
ORDER BY total_revenue DESC;

## Phase 7: Schema Verification

In [ ]:
%%sql -r verify_tables
-- Verify all tables and row counts
SELECT TABLE_NAME, ROW_COUNT
FROM RETAIL_SNOWFLAKE_DW.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'SNOWFLAKE_SCHEMA'
ORDER BY TABLE_NAME;

In [ ]:
%%sql -r verify_fk
-- Verify Foreign Key Relationships
SELECT tc.TABLE_NAME AS child_table, tc.CONSTRAINT_NAME, ccu.TABLE_NAME AS parent_table
FROM RETAIL_SNOWFLAKE_DW.INFORMATION_SCHEMA.TABLE_CONSTRAINTS tc
JOIN RETAIL_SNOWFLAKE_DW.INFORMATION_SCHEMA.CONSTRAINT_COLUMN_USAGE ccu
    ON tc.CONSTRAINT_NAME = ccu.CONSTRAINT_NAME
WHERE tc.CONSTRAINT_TYPE = 'FOREIGN KEY'
    AND tc.TABLE_SCHEMA = 'SNOWFLAKE_SCHEMA'
ORDER BY child_table;